# 01 — Ingestion & Bronze Layer

Download OpenF1 data, save **raw Bronze JSONL**, and generate ingestion evidence (manifest, row counts, schema reports).

| Endpoint | Role |
|----------|------|
| `session_result` | **Required** for Gold target `points_finish` |
| `starting_grid` | **Optional** (may be 404 / empty) |
| Other session endpoints | Per-race telemetry and results |
| `meetings`, `sessions` | Global calendar discovery |

**Run order:** `SMOKE_TEST=True` first → then full run with `SMOKE_TEST=False` and `USE_GOOGLE_DRIVE=True`.

> Full ingestion can take **hours**. Use Drive persistence. Official MBA evidence must come from **Colab Pro Plus**.


## Colab setup (required every session)

This cell clones/updates the repo, installs `requirements.txt` and `pip install -e .`, mounts Drive, and sets `OPENF1_DATA_ROOT`. Do not import `openf1_pipeline` before this cell completes.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# A. Repository (code)
REPO_URL = "https://github.com/dk546/openf1-big-data-pipeline.git"
REPO_NAME = "openf1-big-data-pipeline"
PROJECT_ROOT = Path(f"/content/{REPO_NAME}")

# B. Google Drive persistence (outputs) — set OPENF1_DATA_ROOT before importing config
USE_GOOGLE_DRIVE = True
DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/openf1_big_data_pipeline")

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    os.environ["OPENF1_DATA_ROOT"] = str(DRIVE_OUTPUT_ROOT)
    print("OPENF1_DATA_ROOT:", os.environ["OPENF1_DATA_ROOT"])
else:
    os.environ.pop("OPENF1_DATA_ROOT", None)
    print("OPENF1_DATA_ROOT not set — repo-local outputs.")

# D. Clone or update repository
if not PROJECT_ROOT.exists():
    print("Cloning repository...")
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)
else:
    print("Updating repository...")
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull"], check=False)

os.chdir(PROJECT_ROOT)
print("Working directory:", Path.cwd())

# E. Verify project files
_checks = {
    "README.md": PROJECT_ROOT / "README.md",
    "pyproject.toml": PROJECT_ROOT / "pyproject.toml",
    "src/openf1_pipeline": PROJECT_ROOT / "src" / "openf1_pipeline",
    "src/openf1_pipeline/__init__.py": PROJECT_ROOT / "src" / "openf1_pipeline" / "__init__.py",
    "src/openf1_pipeline/config.py": PROJECT_ROOT / "src" / "openf1_pipeline" / "config.py",
}
for name, path in _checks.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing {name}: {path}")

# F. Install dependencies and editable package
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

# G. Fallback: src on sys.path
_src = PROJECT_ROOT / "src"
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

# H. Import test
import openf1_pipeline  # noqa: E402

from openf1_pipeline.config import (  # noqa: E402
    ensure_project_directories,
    get_artifacts_dir,
    get_bronze_dir,
    get_data_dir,
    get_gold_dir,
    get_output_root,
    get_project_root,
    get_reports_dir,
    get_silver_dir,
)

paths = ensure_project_directories()

# I. Path summary
print("PROJECT_ROOT:", get_project_root())
print("OUTPUT_ROOT:", get_output_root())
print("DATA_DIR:", get_data_dir())
print("BRONZE_DIR:", get_bronze_dir())
print("SILVER_DIR:", get_silver_dir())
print("GOLD_DIR:", get_gold_dir())
print("REPORTS_DIR:", get_reports_dir())
print("ARTIFACTS_DIR:", get_artifacts_dir())



## Start Spark (PySpark — local session, no Databricks)


In [ ]:
from openf1_pipeline.utils.spark import get_spark

spark = get_spark()
print("Spark version:", spark.version)


## Bronze configuration & path check


In [ ]:
from openf1_pipeline.config import (
    ENDPOINTS,
    GLOBAL_ENDPOINTS,
    SESSION_ENDPOINTS,
    SEASONS,
    get_bronze_dir,
    get_data_quality_reports_dir,
    get_manifests_dir,
    get_output_root,
    get_project_root,
    get_schemas_dir,
)
from openf1_pipeline.ingestion.ingest import run_bronze_ingestion, summarize_manifest
from openf1_pipeline.bronze.build_bronze import generate_bronze_reports

print("PROJECT_ROOT:", get_project_root())
print("OUTPUT_ROOT:", get_output_root())
print("ENDPOINTS:", ENDPOINTS)
print("GLOBAL_ENDPOINTS:", GLOBAL_ENDPOINTS)
print("SESSION_ENDPOINTS:", SESSION_ENDPOINTS)
print("BRONZE_DIR:", get_bronze_dir())
print("MANIFESTS_DIR:", get_manifests_dir())
print("DATA_QUALITY_REPORTS_DIR:", get_data_quality_reports_dir())
print("SCHEMAS_DIR:", get_schemas_dir())
print("session_result: REQUIRED for Gold points_finish")
print("starting_grid: OPTIONAL — failures are logged and ingestion continues")


In [ ]:
# Step 1: smoke test. Step 2: set SMOKE_TEST = False for full 2023–2025 evidence.
SMOKE_TEST = True
MAX_SESSIONS = 2 if SMOKE_TEST else None
INGEST_SEASONS = [2024] if SMOKE_TEST else SEASONS

print(f"SMOKE_TEST={SMOKE_TEST}, seasons={INGEST_SEASONS}, max_sessions={MAX_SESSIONS}")
if not SMOKE_TEST:
    print("WARNING: Full ingestion may take a long time. Ensure USE_GOOGLE_DRIVE=True above.")


## Run Bronze ingestion


In [ ]:
manifest_df = run_bronze_ingestion(
    seasons=INGEST_SEASONS,
    endpoints=ENDPOINTS,
    bronze_dir=get_bronze_dir(),
    manifests_dir=get_manifests_dir(),
    max_sessions=MAX_SESSIONS,
)

manifest_summary = summarize_manifest(manifest_df)
manifest_summary


## Endpoint status summary


In [ ]:
print("=== Manifest status counts ===")
print(manifest_summary["status_counts"])
print("\n=== Success endpoints ===")
print(manifest_summary["success_endpoints"])
print("\n=== Failed endpoints (continued) ===")
print(manifest_summary["failed_endpoints"])
print("\n=== Row counts by endpoint ===")
for ep, n in sorted(manifest_summary["row_counts_by_endpoint"].items()):
    print(f"  {ep}: {n}")
print(f"\nsession_result rows: {manifest_summary['session_result_total_rows']}")
print(f"starting_grid rows: {manifest_summary['starting_grid_total_rows']}")
if manifest_summary["session_result_total_rows"] == 0 and not SMOKE_TEST:
    print("WARNING: session_result has zero rows — check manifest before Silver/Gold.")
manifest_df.groupby(["endpoint", "status"]).size().unstack(fill_value=0)


## Generate Bronze evidence reports (Spark-first)


In [ ]:
BRONZE_REPORT_ENGINE = "spark"  # fallback: "pandas"

report_result = generate_bronze_reports(
    bronze_dir=get_bronze_dir(),
    data_quality_reports_dir=get_data_quality_reports_dir(),
    schemas_dir=get_schemas_dir(),
    engine=BRONZE_REPORT_ENGINE,
    spark=spark,
)
report_result


## DuckDB validation (Bronze CSV evidence)


In [ ]:
from openf1_pipeline.analytics.duckdb_validation import (
    save_duckdb_validation_reports,
    validate_bronze_with_duckdb,
)

bronze_duckdb = validate_bronze_with_duckdb(get_data_quality_reports_dir())
duckdb_bronze_paths = save_duckdb_validation_reports(
    bronze_duckdb, get_data_quality_reports_dir(), prefix="bronze"
)
duckdb_bronze_paths


## Inspect outputs


In [ ]:
import pandas as pd

row_counts = pd.read_csv(report_result["paths"]["bronze_row_counts"])
schema_report = pd.read_csv(report_result["paths"]["bronze_schema_report"])
schema_drift = pd.read_csv(report_result["paths"]["bronze_schema_drift"])

print("Row counts by endpoint:")
display(row_counts.groupby("endpoint")["row_count"].sum().reset_index())
display(schema_report.head(10))
drift_flags = schema_drift[schema_drift["possible_schema_drift_flag"] == True]
print(f"Schema drift flags: {len(drift_flags)}")
if len(drift_flags):
    display(drift_flags.head(15))
